In [ ]:
from pathlib import Path
import random
import numpy as np
import rasterio
import matplotlib.pyplot as plt

PATCH_DIR = Path("/home/silvia/Desktop/GIGI/ASI_WGD_2026_Myanmar/BDD/data/patches")

N_DAMAGED = 5
N_INTACT = 5
SEED = np.random.randint(0, 10000)

def read_tif(path):
    with rasterio.open(path) as src:
        return src.read()

def normalize_percentile(arr, pmin=2, pmax=98):
    arr = arr.astype(np.float32)
    out = np.zeros_like(arr)

    for c in range(arr.shape[0]):
        lo, hi = np.percentile(arr[c], [pmin, pmax])
        out[c] = np.clip((arr[c] - lo) / (hi - lo + 1e-6), 0, 1)

    return out

def make_rgb_vis(rgb):
    return normalize_percentile(rgb[:3]).transpose(1, 2, 0)

def make_sar_vis(sar):
    sar = normalize_percentile(sar[:2])
    ch1, ch2 = sar[0], sar[1]
    diff = ch1 - ch2
    diff = (diff - diff.min()) / (diff.max() - diff.min() + 1e-6)
    return np.stack([ch1, ch2, diff], axis=-1)

def overlay_mask_on_rgb(rgb_vis, mask, alpha=0.45):
    overlay = rgb_vis.copy()
    m = mask > 0
    overlay[m, 0] = (1 - alpha) * overlay[m, 0] + alpha
    overlay[m, 1] = (1 - alpha) * overlay[m, 1]
    overlay[m, 2] = (1 - alpha) * overlay[m, 2]
    return overlay

sar_files = sorted(PATCH_DIR.glob("*_SAR.tif"))

sample_ids = [
    p.name.replace("_SAR.tif", "")
    for p in sar_files
    if not p.name.endswith("_SARftp.tif")
]

damaged_ids = [sid for sid in sample_ids if sid.startswith("D_")]
intact_ids = [sid for sid in sample_ids if sid.startswith("I_")]

print(f"Damaged patches: {len(damaged_ids)}")
print(f"Intact patches: {len(intact_ids)}")

random.seed(SEED)

selected_damaged = random.sample(damaged_ids, min(N_DAMAGED, len(damaged_ids)))
selected_intact = random.sample(intact_ids, min(N_INTACT, len(intact_ids)))

selected_ids = selected_damaged + selected_intact
random.shuffle(selected_ids)

selected_ids

In [ ]:
n = len(selected_ids)
fig, axes = plt.subplots(n, 5, figsize=(17, 3.2 * n))

if n == 1:
    axes = np.expand_dims(axes, axis=0)

for row, sample_id in enumerate(selected_ids):
    label_name = "damaged" if sample_id.startswith("D_") else "intact"

    sar_path = PATCH_DIR / f"{sample_id}_SAR.tif"
    ftp_path = PATCH_DIR / f"{sample_id}_SARftp.tif"
    rgb_path = PATCH_DIR / f"{sample_id}_RGB.tif"

    sar = read_tif(sar_path)
    ftp = read_tif(ftp_path)[0]
    rgb = read_tif(rgb_path)

    rgb_vis = make_rgb_vis(rgb)
    rgb_overlay = overlay_mask_on_rgb(rgb_vis, ftp)

    sar_norm = normalize_percentile(sar[:2])
    vv_vis = sar_norm[0]
    vh_vis = sar_norm[1]

    axes[row, 0].imshow(rgb_vis)
    axes[row, 0].set_title(f"{sample_id}\nRGB - {label_name}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(rgb_overlay)
    axes[row, 1].set_title("RGB + footprint")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(vv_vis, cmap="gray")
    axes[row, 2].set_title("SAR band 1 / VV")
    axes[row, 2].axis("off")

    axes[row, 3].imshow(vh_vis, cmap="gray")
    axes[row, 3].set_title("SAR band 2 / VH")
    axes[row, 3].axis("off")

    axes[row, 4].imshow(ftp, cmap="gray")
    axes[row, 4].set_title("Footprint mask")
    axes[row, 4].axis("off")

plt.tight_layout()
plt.show()